# SE1_data_access

This notebook documents reproducible access to a subset of public raw datasets used in the HydroReg-Oulujoki project. Direct-download files are stored outside the repository, and one public FMI API request is preserved as a raw response file for transparency and reproducibility.

## Goals
- download public raw datasets from stable URLs
- store raw files outside the Git repository
- document one public API request for reproducibility
- keep raw-data access separate from preprocessing and modelling

This notebook focuses on **data access only**.
It does not run the full hydrological or hydropower workflow.


In [ ]:
## Imports


In [1]:
from pathlib import Path
import requests
import os
import time
from urllib.parse import urlparse
import pandas as pd

## Create Raw data folder

In [9]:
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "README.md").exists():
            return p
    return start

notebook_dir = Path.cwd()
repo_root = find_repo_root(notebook_dir)
external_base = repo_root.parent

raw_data_dir = external_base / "HydroReg-Oulujoki_rawdata"
direct_dir = raw_data_dir / "direct_downloads"
api_dir = raw_data_dir / "api"

direct_dir.mkdir(parents=True, exist_ok=True)
api_dir.mkdir(parents=True, exist_ok=True)

print("Notebook directory :", notebook_dir)
print("Repository root    :", repo_root)
print("Raw data directory :", raw_data_dir)

Notebook directory : /home/jovyan/WaterDig/ClassData
Repository root    : /home/jovyan/WaterDig
Raw data directory : /home/jovyan/HydroReg-Oulujoki_rawdata


## Data description

This notebook downloads a small but representative subset of public raw datasets required for the HydroReg-Oulujoki workflow.

### Data groups included here
1. **Direct-download spatial datasets**
   - catchments
   - land cover

2. **Public API example**
   - FMI hourly weather observations for Oulu

### Purpose
- spatial datasets: support basin/sub-catchment and HRU-related preprocessing
- FMI observations: support hydrological forcing preparation

### Scope
This notebook demonstrates **reproducible raw-data access** only.
It does not perform:
- spatial preprocessing
- HBV calibration
- FlexTool scheduling
- scenario analysis

## How to run this notebook

### Inputs
- stable public URLs from `datalinks.txt`
- one public FMI API request

### Outputs
Raw files are stored **outside the Git repository** in:

`../HydroReg-Oulujoki_rawdata/`

Subfolders:
- `direct_downloads/`
- `api/`

### Storage requirements
Storage demand is low to moderate for this notebook.
The largest files are the land-cover zip datasets.

In [10]:
!df -k ..

Filesystem     1K-blocks    Used Available Use% Mounted on
/dev/sdb        10218772 5765228   4437160  57% /home/jovyan


## Helper functions

In [5]:
def get_filename_from_url(url):
    path = urlparse(url).path
    name = os.path.basename(path)
    return name if name else "downloaded_file"

def download_file(url, output_dir, timeout=60):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    filename = get_filename_from_url(url)
    filepath = output_dir / filename

    if filepath.exists():
        print(f"File already exists, skipping: {filename}")
        return filepath

    try:
        print(f"Downloading: {url}")
        with requests.get(url, stream=True, timeout=timeout) as r:
            r.raise_for_status()
            with open(filepath, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
        print(f"Saved: {filepath}")
        return filepath

    except Exception as e:
        print(f"Failed: {url}")
        print(f"Error: {e}")
        return None

## Links

In [11]:
possible_link_files = [
    repo_root / "inputs" / "datalinks.txt",
    repo_root / "datalinks.txt",
    notebook_dir / "datalinks.txt",
]

links_file = None
for p in possible_link_files:
    if p.exists():
        links_file = p
        break

if links_file is None:
    raise FileNotFoundError("Could not find datalinks.txt")

print("Using links file:", links_file)

with open(links_file, "r", encoding="utf-8") as f:
    all_links = [line.strip() for line in f if line.strip()]

print(f"Found {len(all_links)} links in datalinks.txt")
pd.DataFrame({"url": all_links})

Using links file: /home/jovyan/WaterDig/ClassData/datalinks.txt
Found 74 links in datalinks.txt


,url
0,# HydroReg-Oulujoki dataset links
1,# Concrete official URLs and example API queri...
2,# Replace dates / tokens / station IDs as needed.
3,# Do not store private tokens in this file.
4,# --------------------------------------------...
...,...
69,# --------------------------------------------...
70,# - ENTSO-E token must be kept in an environme...
71,# - SYKE/FMI queries can be narrowed by statio...
72,"# - For full reproducibility, create a second ..."


## Direct-download files vs API endpoints vs reference webpages

The links file contains mixed resource types:
- direct downloadable files (for example `.zip`)
- API/service endpoints
- reference webpages

For reproducible raw-data access, these resource types should be handled separately.
This notebook automatically downloads only the **direct file URLs** and stores the **FMI API response** separately.

In [12]:
direct_urls = [u for u in all_links if u.lower().endswith(".zip")]

print("Direct-download URLs:")
for u in direct_urls:
    print("-", u)

downloaded_direct = []
for url in direct_urls:
    fp = download_file(url, direct_dir)
    downloaded_direct.append(fp)
    time.sleep(2)

downloaded_direct

Direct-download URLs:
- SYKE_CATCHMENT_DIVISION_ZIP: https://wwwd3.ymparisto.fi/d3/gis_data/spesific/valumaalueet.zip
- SYKE_CORINE_2018_RASTER_20M_ZIP: https://wwwd3.ymparisto.fi/d3/Static_rs/spesific/clc2018_fi20m.zip
- SYKE_CORINE_2018_VECTOR_25HA_ZIP: https://wwwd3.ymparisto.fi/d3/Static_rs/spesific/clc2018eu25ha.zip
File already exists, skipping: valumaalueet.zip
File already exists, skipping: clc2018_fi20m.zip
File already exists, skipping: clc2018eu25ha.zip


[PosixPath('/home/jovyan/HydroReg-Oulujoki_rawdata/direct_downloads/valumaalueet.zip'),
 PosixPath('/home/jovyan/HydroReg-Oulujoki_rawdata/direct_downloads/clc2018_fi20m.zip'),
 PosixPath('/home/jovyan/HydroReg-Oulujoki_rawdata/direct_downloads/clc2018eu25ha.zip')]

## API 

In [15]:
fmi_query = (
    "https://opendata.fmi.fi/wfs"
    "?service=WFS"
    "&version=2.0.0"
    "&request=getFeature"
    "&storedquery_id=fmi::observations::weather::hourly::timevaluepair"
    "&place=oulu"
    "&starttime=2024-01-01T00:00:00Z"
    "&endtime=2024-01-03T00:00:00Z"
    "&timestep=60"
)

response = requests.get(fmi_query, timeout=90)

print("Status:", response.status_code)
print(response.text[:1000])   

response.raise_for_status()

fmi_outfile = api_dir / "fmi_oulu_2024-01-01_2024-01-03_hourly.xml"
with open(fmi_outfile, "wb") as f:
    f.write(response.content)

print("Saved:", fmi_outfile)

Status: 200
<?xml version="1.0" encoding="UTF-8"?>
<wfs:FeatureCollection
    timeStamp="2026-03-07T09:52:20Z"
    numberMatched="12"
    numberReturned="12"
           xmlns:wfs="http://www.opengis.net/wfs/2.0" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
        xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:om="http://www.opengis.net/om/2.0"
        xmlns:ompr="http://inspire.ec.europa.eu/schemas/ompr/3.0"
        xmlns:omso="http://inspire.ec.europa.eu/schemas/omso/3.0"
        xmlns:gml="http://www.opengis.net/gml/3.2" xmlns:gmd="http://www.isotc211.org/2005/gmd"
        xmlns:gco="http://www.isotc211.org/2005/gco" xmlns:swe="http://www.opengis.net/swe/2.0"
        xmlns:gmlcov="http://www.opengis.net/gmlcov/1.0"
        xmlns:sam="http://www.opengis.net/sampling/2.0"
        xmlns:sams="http://www.opengis.net/samplingSpatial/2.0"
        xmlns:wml2="http://www.opengis.net/waterml/2.0"
	xmlns:target="http://xml.fmi.fi/namespace/om/atmosphericfeatures/1.1"
        xsi:sc

## Report

In [14]:
files = []
for p in raw_data_dir.rglob("*"):
    if p.is_file():
        files.append({
            "file": str(p),
            "size_mb": round(p.stat().st_size / 1024 / 1024, 2)
        })

files_df = pd.DataFrame(files).sort_values("file").reset_index(drop=True)
files_df

,file,size_mb
0,/home/jovyan/HydroReg-Oulujoki_rawdata/direct_...,260.36
1,/home/jovyan/HydroReg-Oulujoki_rawdata/direct_...,109.16
2,/home/jovyan/HydroReg-Oulujoki_rawdata/direct_...,245.40


## Notes on authentication and scope

- The direct-download datasets used here are public and do not require authentication.
- The FMI request used here is public and does not require authentication.
- ENTSO-E data are not downloaded in this notebook because token-based access should be documented separately.
- Fortum and Oulun Energia links are reference webpages and are not treated here as machine-readable raw datasets.
- SYKE hydrology API endpoints listed in the links file are also better handled separately with explicit query parameters rather than downloaded blindly as files.

## Reproducibility summary

This notebook demonstrates a reproducible raw-data access pattern by:
1. reading documented public links,
2. separating file downloads from API requests,
3. storing raw outputs outside the repository,
4. preserving the exact API query used.

This supports transparent downstream preprocessing and modelling.

## Zenodo archiving note

For this project, a practical Zenodo strategy is:

- archive the repository snapshot
- archive processed/shareable outputs
- archive **lightweight provenance files** such as `download_manifest.jsonl`
- only publish raw data

## What you should have after running

After running the relevant cells, you should have:

- downloaded raw files in `~/raw_data/<project_name>/`
- a provenance file `download_manifest.jsonl`
- a synthetic test dataset in `~/raw_data/<project_name>/synthetic/`
- an optional API example query printed in the notebook, and an API output file if you decide to run that cell
